In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
create connection if not exists youtube_earthquake_test
type HTTP
options (
    host = 'https://earthquake.usgs.gov',
    port 443,
    base_path = '/earthquakes/feed/v1.0/',
    bearer_token = 'na'
)

-- https://earthquake.usgs.gov//earthquakes/feed/v1.0/summary/all_day.geojson

In [0]:
%sql
-- create volume if not exists youtube_dev.bronze.earthquake_data_vol;

**dbutils.widgets is used for:** 
You can quickly switch your code between _youtube_dev_, _youtube_stage_, or _youtube_prod_ environments without editing the actual script.

In [0]:

dbutils.widgets.text('catalog_name', 'youtube_dev','youtube_dev')
catalog_name = dbutils.widgets.get('catalog_name')
print(catalog_name)


In [0]:
%py
spark.sql(
    f"use catalog {catalog_name}"
)
spark.sql(
    "use schema bronze"
)
spark.sql("create volume if not exists earthquake_data_vol");

In [0]:
import requests
import json
import datetime

url = "https://earthquake.usgs.gov//earthquakes/feed/v1.0/summary/all_day.geojson"
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Error {response.status_code} while fetching data from {url}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data_vol/earthquake_data_vol_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)